# [Sensitivity Analysis] The Thermodynamic Starvation (KAN-TabNet) | Step LR | Poker Hand

**Optimized Reference Configuration:** `n_d=6, n_a=4`, `kan_grid_size=6`, `kan_spline_order=2`, `initial_lr=0.005`

### Methodological Context: The Control Variables
To precisely isolate the effects of the optimization thermodynamics, this sensitivity analysis uses the exact same architectural pipeline (`n_d=6, n_a=4`), StepLR scheduler, grid resolution (`G=6`), and spline order (`k=2`) as our optimized reference configuration.

By fixing the structural capacity and continuous mathematical geometries of the network, we strictly guarantee that any performance variations observed in this notebook are purely the result of the initial kinetic energy provided to the optimizer.

### The Hypothesis
In this study, we evaluate a "cold start" optimization environment by reducing the initial learning rate to `0.0025` (half the energy of our reference configuration).

We hypothesize that learning the rigid, discrete combinatorial logic of playing cards requires a specific threshold of thermodynamic energy to escape local minima. Because quadratic splines ($k=2$) are inherently sharp, an insufficient learning rate will cause the splines to prematurely "freeze" into suboptimal integer bins during the early exploration phase. This notebook serves to empirically evaluate whether this thermodynamic starvation prevents the network from executing the necessary evolutionary leaps required to deduce complex, rare patterns like Flushes and Full Houses, thereby validating the necessity of the aggressive `0.005` reference learning rate.

### Setup

In [1]:
%%capture
# install TabNet fork which allows switching between vanilla TabNet and KAN-TabNet
!pip install git+https://github.com/chuo-v/tabnet.git@v1.0.1-kan

In [2]:
import os
import json
import numpy as np
import random
import torch

seed = 11
random.seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [3]:
import warnings

warnings.filterwarnings("ignore", category=UserWarning)

### Dataset Fetching

In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
from pytorch_tabnet.tab_model import TabNetClassifier
import gc

url_train = 'https://archive.ics.uci.edu/ml/machine-learning-databases/poker/poker-hand-training-true.data'
url_test = 'https://archive.ics.uci.edu/ml/machine-learning-databases/poker/poker-hand-testing.data'

df_train = pd.read_csv(url_train, header=None)
df_test = pd.read_csv(url_test, header=None)

# 0-index the categorical features for PyTorch embedding layers
X_train_full = df_train.iloc[:, :-1].values - 1
y_train_full = df_train.iloc[:, -1].values.astype(int)

X_test = df_test.iloc[:, :-1].values - 1
y_test = df_test.iloc[:, -1].values.astype(int)

# remove the permutation invariance trap that cripples standard MLPs on this dataset
def sort_poker_hands(X):
    # reshape from (N, 10) to (N, 5, 2) to lock each Suit with its Rank
    X_pairs = X.reshape(-1, 5, 2)

    # get the indices that sort each hand based on the Rank (index 1 of the pair)
    sort_indices = np.argsort(X_pairs[:, :, 1], axis=1)

    # apply the sort to both Suit and Rank simultaneously
    row_indices = np.arange(X.shape[0])[:, np.newaxis]
    X_sorted_pairs = X_pairs[row_indices, sort_indices]

    # flatten back to the standard (N, 10) TabNet input format
    return X_sorted_pairs.reshape(-1, 10)

print("Sorting hands to remove permutation invariance...")
X_train_full = sort_poker_hands(X_train_full)
X_test = sort_poker_hands(X_test)
print("Sorting complete.")

del df_train, df_test
gc.collect()

# split 6k samples for validation from the 25k training dataset
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full, test_size=6000, random_state=seed, stratify=y_train_full
)

print(f"Train shape: {X_train.shape}")
print(f"Valid shape: {X_valid.shape}")
print(f"Test shape:  {X_test.shape}")

Sorting hands to remove permutation invariance...
Sorting complete.
Train shape: (19010, 10)
Valid shape: (6000, 10)
Test shape:  (1000000, 10)


### Model Training

In [5]:
MAX_EPOCHS = 10000

In [6]:
# define categorical structures for the 10 input columns
categorical_indices = list(range(10))
categorical_dimensions = [4, 13, 4, 13, 4, 13, 4, 13, 4, 13]
categorical_embedding_dims = [1] * 10

# Hyperparameters from original paper.
# Notes:
# - momentum is 1 - 0.95 (paper m_B) to align with PyTorch's BatchNorm API.
# - scheduler step_size of 100 epochs approximates the paper's 500 iterations
#   (19k train rows / 4096 batch size = ~4.64 iters/epoch. 500 / 4.64 = ~107. Using 100).
paper_config = {
    'n_steps': 4,
    'gamma': 1.5,
    'n_independent': 2,
    'n_shared': 2,
    'lambda_sparse': 1e-6,
    'momentum': 0.05,
    'optimizer_fn': torch.optim.Adam,
    'scheduler_fn': torch.optim.lr_scheduler.StepLR,
    'scheduler_params': dict(step_size=100, gamma=0.95),
    'mask_type': 'sparsemax'
}

clf_kan = TabNetClassifier(
    n_d=6,
    n_a=4,
    **paper_config,
    cat_idxs=categorical_indices,
    cat_dims=categorical_dimensions,
    cat_emb_dim=categorical_embedding_dims,
    clip_value=2.,
    optimizer_params=dict(lr=0.0025),
    use_kan=True,
    kan_grid_size=6,
    kan_spline_order=2,
    seed=seed,
    verbose=250
)

In [7]:
# Hyperparameters from original paper.
paper_fit_config = {
    'batch_size': 4096,
    'virtual_batch_size': 1024,
}

clf_kan.fit(
    X_train=X_train, y_train=y_train,
    eval_set=[(X_valid, y_valid)],
    eval_name=['valid'],
    eval_metric=['accuracy'],
    **paper_fit_config,
    max_epochs=MAX_EPOCHS,
    patience=MAX_EPOCHS,
    num_workers=0,
    drop_last=False,
    compute_importance=False
)

epoch 0  | loss: 1.87663 | valid_accuracy: 0.49867 |  0:00:01s
epoch 250| loss: 0.46182 | valid_accuracy: 0.83967 |  0:02:02s
epoch 500| loss: 0.18477 | valid_accuracy: 0.92883 |  0:04:02s
epoch 750| loss: 0.11365 | valid_accuracy: 0.952   |  0:06:01s
epoch 1000| loss: 0.12028 | valid_accuracy: 0.9465  |  0:08:00s
epoch 1250| loss: 0.06397 | valid_accuracy: 0.96233 |  0:09:58s
epoch 1500| loss: 0.05543 | valid_accuracy: 0.96533 |  0:11:56s
epoch 1750| loss: 0.05168 | valid_accuracy: 0.96717 |  0:13:53s
epoch 2000| loss: 0.04071 | valid_accuracy: 0.96767 |  0:15:51s
epoch 2250| loss: 0.03259 | valid_accuracy: 0.97317 |  0:17:49s
epoch 2500| loss: 0.03358 | valid_accuracy: 0.97317 |  0:19:46s
epoch 2750| loss: 0.02261 | valid_accuracy: 0.97533 |  0:21:44s
epoch 3000| loss: 0.02071 | valid_accuracy: 0.97567 |  0:23:41s
epoch 3250| loss: 0.0204  | valid_accuracy: 0.97533 |  0:25:39s
epoch 3500| loss: 0.01745 | valid_accuracy: 0.9765  |  0:27:37s
epoch 3750| loss: 0.01668 | valid_accuracy: 

In [8]:
# sum up all learnable weights in the PyTorch network
param_count = sum(p.numel() for p in clf_kan.network.parameters() if p.requires_grad)

print(f"Total Learnable Parameters: {param_count:,}")

Total Learnable Parameters: 25,225


### Test Evaluation

In [9]:
from sklearn.metrics import accuracy_score

# evaluate on the hold-out test set
preds = clf_kan.predict(X_test)
test_acc = accuracy_score(y_test, preds)

print(f"Test Accuracy: {test_acc:.5f}")

Test Accuracy: 0.97677


### Data Export

In [10]:
BASE_DIR = './kan-tabnet-experiments'
TABLES_DIR = f'{BASE_DIR}/results/poker_hand/tables'
MODELS_DIR = f'{BASE_DIR}/results/poker_hand/models'

os.makedirs(TABLES_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

# package all metrics into a single JSON payload
experiment_results = {
    "model_type": "KAN-TabNet",
    "dataset": "Poker Hand",
    "parameters": param_count,
    "scheduler": "StepLR",
    "final_test_accuracy": test_acc,
    "best_epoch": clf_kan.best_epoch,
    "training_history": clf_kan.history.history
}

# save JSON metrics
JSON_FILEPATH = f'{TABLES_DIR}/07_kan_sensitivity_lr_0025_metrics.json'
with open(JSON_FILEPATH, 'w') as f:
    json.dump(experiment_results, f, indent=4)
print(f"Metrics successfully saved to {JSON_FILEPATH}")

# save the model weights
_ = clf_kan.save_model(f'{MODELS_DIR}/07_kan_sensitivity_lr_0025_{param_count // 1000}k')

Metrics successfully saved to ./kan-tabnet-experiments/results/poker_hand/tables/07_kan_sensitivity_lr_0025_metrics.json
Successfully saved model at ./kan-tabnet-experiments/results/poker_hand/models/07_kan_sensitivity_lr_0025_25k.zip
